# Qwen Math GRPO/DAPO — 七组消融（Kaggle T4 一键运行）

从零手写的 GRPO 训练循环 + DAPO 四项单变量开关 + 熵正则扩展，在 **Qwen2.5-0.5B-Instruct + LoRA** 上跑 GSM8K 数学推理 RL。

## 使用方法（共 3 步）

1. Kaggle → **New Notebook** → File → **Import Notebook**，上传本文件
2. 右侧 Settings → Accelerator → **GPU T4 x1**（免费）
3. **Run All**，等跑完（约 5-8 小时）

> 想先快速验证？把 Cell 6 里的 `MAX_STEPS` 改成 `5`（约 40 分钟跑完七组）。

> **每个 cell 都会输出完整日志和报错，绝不吞任何信息。如果某个 cell 失败了，直接看输出就是完整 traceback。**

## 跑完产出（/kaggle/working/ 下，可下载）

- `ablation_results.json` — 每组最终指标
- `ablation_summary.png` — 七组 reward 训练曲线 + 最终 accuracy 对比图

In [ ]:
# Cell 1: GPU 检查
!nvidia-smi
import torch
assert torch.cuda.is_available(), "请在右侧 Settings -> Accelerator 选择 GPU T4 后重新运行"
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# Cell 2: clone 仓库 + 安装依赖 + 验证文件完整性
import os, subprocess, sys

os.environ["HF_ENDPOINT"] = "https://huggingface.co"
os.environ["PYTHONUNBUFFERED"] = "1"

# ---- git clone ----
print("=== git clone ===")
result = subprocess.run(["git", "clone", "https://github.com/rayyy032/qwen-math-grpo-dapo.git"],
                       capture_output=True, text=True)
print("stdout:", result.stdout)
if result.stderr:
    print("stderr:", result.stderr)
print("returncode:", result.returncode)
assert result.returncode == 0, f"git clone 失败: {result.stderr}"

%cd /kaggle/working/qwen-math-grpo-dapo

# ---- 检查关键文件是否存在 ----
critical_files = [
    "scripts/train.py",
    "src/config.py", "src/rewards.py", "src/losses.py",
    "src/generation.py", "src/data.py", "src/evaluation.py",
    "src/diagnostics.py", "src/train_utils.py",
    "src/agents/base_agent.py", "src/agents/grpo_agent.py",
    "tests/test_reward.py",
]
print("\n=== 检查关键文件 ===")
all_exist = True
for f in critical_files:
    exists = os.path.isfile(f)
    size = os.path.getsize(f) if exists else 0
    status = "OK" if exists and size > 0 else "MISSING/EMPTY"
    print(f"  [{status}] {f} ({size} bytes)")
    if not exists or size == 0:
        all_exist = False
assert all_exist, "有文件缺失！检查 git clone 是否成功"

# ---- pip install ----
print("\n=== pip install ===")
result = subprocess.run([sys.executable, "-m", "pip", "install",
    "transformers>=4.45", "datasets", "peft", "accelerate",
    "math-verify", "swanlab", "pyyaml", "tqdm"],
    capture_output=True, text=True)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.stderr:
    print("stderr:", result.stderr[-2000:])
print("returncode:", result.returncode)

print("\n== 依赖安装完成 ==")

# ---- 验证关键 import ----
print("\n=== 验证 import ===")
for mod in ["torch", "transformers", "peft", "datasets", "numpy", "yaml"]:
    try:
        m = __import__(mod)
        v = getattr(m, "__version__", "?")
        print(f"  [OK] {mod} {v}")
    except Exception as e:
        print(f"  [FAIL] {mod}: {e}")

try:
    import math_verify
    print("  [OK] math_verify")
except Exception as e:
    print(f"  [FAIL] math_verify: {e}")
    print("  (不影响核心训练，只影响数值等价检测的 fallback)")

In [ ]:
# Cell 3: 27 项边界单测（verbose 模式，完整输出）
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"],
                       capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print("returncode:", result.returncode)
if result.returncode != 0:
    print("\n!! 有测试失败！上面的输出包含完整 traceback，请截图")
    raise RuntimeError("Tests failed — see output above")

In [ ]:
# Cell 4: ~1 分钟冒烟测试 — 验证 GPU 上整条流水线
# 如果这步失败，后面的七组消融肯定也跑不通。
# 完整 stdout + stderr + traceback 会直接输出在下面。
import subprocess, sys, os
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
proc = subprocess.run(
    [sys.executable, "scripts/train.py",
     "--algorithm", "dapo", "--dataset", "synthetic",
     "--train-samples", "12", "--eval-samples", "4", "--max-steps", "1",
     "--group-size", "4", "--groups-per-step", "1", "--max-new-tokens", "48",
     "--device", "cuda", "--use-swanlab", "false"],
    capture_output=True, text=True, env=env)
print("=== STDOUT ===")
print(proc.stdout)
if proc.stderr:
    print("=== STDERR ===")
    print(proc.stderr)
print("=== returncode:", proc.returncode, "===")
if proc.returncode != 0:
    print("\n!! 冒烟测试失败！上面的完整 traceback 就是根因。请截图。")
    raise RuntimeError("Smoke test failed — see full traceback above")
else:
    print("\nOK 冒烟测试通过，可以继续跑七组消融")

## 七组单变量消融

下面的 cell 会依次跑 7 组训练。**每组实时输出 stdout**（每步的 reward/KL/熵/hack rates）。

如果某组失败：**完整 stderr/traceback 会直接输出在 `=== [STDERR] preset: xxx ===` 下面**，然后自动跳到下一组继续，不会中断整个 notebook。

想快速验证就把 `MAX_STEPS` 改成 `5`。

In [ ]:
# ============================================================
# Cell 6: 七组单变量消融（唯一需要改的配置区）
# ============================================================
PRESETS = ["grpo", "clip_higher", "dynamic", "token_level", "overlong", "dapo", "entropy_reg"]
MAX_STEPS = 30          # 每组步数；30 步约 25-50 min/组
DATASET = "gsm8k"       # gsm8k | gsm8k_math
TRAIN_SAMPLES = 500
EVAL_SAMPLES = 100
# ============================================================

import subprocess, time, json, sys, os, traceback, io

REPO = "/kaggle/working/qwen-math-grpo-dapo"
results = {}
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

for algo in PRESETS:
    print(f"\n{'='*70}")
    print(f">>> preset: {algo}")
    print(f"{'='*70}", flush=True)

    cmd = [sys.executable, "scripts/train.py",
           "--algorithm", algo, "--dataset", DATASET,
           "--train-samples", str(TRAIN_SAMPLES),
           "--eval-samples", str(EVAL_SAMPLES),
           "--pass-at-k", "8", "--pass-k-samples", "50",
           "--max-steps", str(MAX_STEPS),
           "--device", "cuda", "--run-name", algo]
    t0 = time.time()

    try:
        # 用 Popen + 逐行读 stdout 实现实时输出，stderr 单独收集
        proc = subprocess.Popen(cmd, cwd=REPO,
                                stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE,
                                text=True, bufsize=1, env=env)

        stdout_lines = []
        stderr_lines = []

        # 逐行读 stdout 实时打印
        for line in proc.stdout:
            line = line.rstrip()
            print(line, flush=True)
            stdout_lines.append(line)

        # 读 stderr
        stderr_text = proc.stderr.read()
        if stderr_text:
            print(f"\n=== [STDERR] preset: {algo} ===", flush=True)
            print(stderr_text, flush=True)
            stderr_lines = stderr_text.split("\n")

        proc.wait()
        print(f"=== [returncode] {proc.returncode} ===", flush=True)

        all_lines = stdout_lines + stderr_lines

        if proc.returncode != 0:
            print(f"\n=== [ERROR] preset: {algo} — train.py 退出码 {proc.returncode} ===", flush=True)
            print(f"=== 完整 traceback 见上方 STDERR 输出 ===", flush=True)
            results[algo] = {
                "minutes": round((time.time() - t0) / 60, 1),
                "accuracy": None, "format_rate": None, "pass@8": None,
                "final_reward": None, "surprisal": None,
                "clip_fraction": None, "zero_var_ratio": None,
                "mean_completion_tokens": None, "reward_curve": [],
                "error": f"train.py exit code {proc.returncode}",
                "stderr_tail": "\n".join(stderr_lines[-20:]) if stderr_lines else "",
            }
            with open("/kaggle/working/ablation_results.json", "w") as f:
                json.dump(results, f, indent=2)
            continue

        # ---- 解析该组结果 ----
        curve, final, eval_vals = [], {}, {}
        for line in all_lines:
            line = line.strip()
            if line.startswith("[step"):
                body = line.split("] ", 1)[-1] if "] " in line else ""
                vals = {}
                for part in body.split(" | "):
                    k, _, v = part.partition("=")
                    try:
                        vals[k] = float(v)
                    except ValueError:
                        pass
                final = vals
                if "mean" in vals:
                    curve.append(vals["mean"])
            elif line.startswith("[eval]"):
                for part in line.split("] ", 1)[-1].split(" "):
                    k, _, v = part.partition("=")
                    if k == "accuracy" or k == "format_rate" or k.startswith("pass@"):
                        try:
                            eval_vals[k] = float(v)
                        except ValueError:
                            pass

        results[algo] = {
            "minutes": round((time.time() - t0) / 60, 1),
            "accuracy": eval_vals.get("accuracy"),
            "format_rate": eval_vals.get("format_rate"),
            "pass@8": eval_vals.get("pass@8"),
            "final_reward": final.get("mean"),
            "surprisal": final.get("mean_sampled_token_surprisal"),
            "clip_fraction": final.get("clip_fraction"),
            "zero_var_ratio": final.get("zero_variance_group_ratio"),
            "mean_completion_tokens": final.get("mean_completion_tokens"),
            "reward_curve": curve,
        }
        with open("/kaggle/working/ablation_results.json", "w") as f:
            json.dump(results, f, indent=2)
        print(f"\n[summary] {algo}: acc={eval_vals.get('accuracy')} "
              f"pass@8={eval_vals.get('pass@8')} ({results[algo]['minutes']} min)", flush=True)

    except Exception as e:
        print(f"\n=== [FATAL] preset: {algo} ===", flush=True)
        traceback.print_exc()
        results[algo] = {
            "minutes": round((time.time() - t0) / 60, 1),
            "error": str(e),
            "accuracy": None, "format_rate": None, "pass@8": None,
            "final_reward": None, "surprisal": None,
            "clip_fraction": None, "zero_var_ratio": None,
            "mean_completion_tokens": None, "reward_curve": [],
        }
        with open("/kaggle/working/ablation_results.json", "w") as f:
            json.dump(results, f, indent=2)
        continue

print(f"\n{'='*70}")
print("所有组完成。结果保存在 -> /kaggle/working/ablation_results.json")
print(f"{'='*70}")

In [ ]:
# Cell 7: 画图 + 结果汇总
import json
import pandas as pd
import matplotlib.pyplot as plt

with open("/kaggle/working/ablation_results.json") as f:
    results = json.load(f)

print("=== 完整结果表 ===")
df = pd.DataFrame(results).T
display(df[["accuracy", "pass@8", "format_rate", "final_reward", "surprisal",
            "clip_fraction", "zero_var_ratio", "mean_completion_tokens", "minutes"]])

success = {a: r for a, r in results.items() if r.get("accuracy") is not None}
failed = {a: r for a, r in results.items() if r.get("accuracy") is None}
print(f"\n成功: {len(success)} 组, 失败: {len(failed)} 组")
if failed:
    for a, r in failed.items():
        print(f"  FAIL {a}: {r.get('error', 'unknown')} ({r.get('minutes', 0)} min)")
        if r.get("stderr_tail"):
            print(f"    stderr tail: {r['stderr_tail'][-200:]}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for algo, r in results.items():
    if r.get("reward_curve"):
        axes[0].plot(r["reward_curve"], label=algo, alpha=0.85)
axes[0].set_xlabel("step")
axes[0].set_ylabel("mean reward")
axes[0].set_title("Training reward curves (7 presets)")
if success:
    axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

algos = list(results.keys())
accs = [results[a]["accuracy"] or 0 for a in algos]
colors = ["#9e9e9e"] * len(algos)
if "dapo" in algos:
    colors[algos.index("dapo")] = "#d62728"
if "entropy_reg" in algos:
    colors[algos.index("entropy_reg")] = "#1f77b4"
axes[1].bar(algos, accs, color=colors)
axes[1].set_ylabel("GSM8K accuracy")
axes[1].set_title("Final eval accuracy")
axes[1].tick_params(axis="x", rotation=30)
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("/kaggle/working/ablation_summary.png", dpi=150)
print("saved -> /kaggle/working/ablation_summary.png")
plt.show()

## 如果有组失败了

1. 看 Cell 6 输出中 `=== [STDERR] preset: xxx ===` 下面的完整 traceback
2. 看 `=== [ERROR] preset: xxx ===` 确认退出码
3. 截图 Cell 6 的完整输出发给我

失败的组在 `ablation_results.json` 里会有 `"error"` 和 `"stderr_tail"` 字段记录原因。